# Machine Learning Based Network Intrusion Detection System
### IBM SkillsBuild Cyber Security 2026

**Student:** Ishwar Dnyaneshwar Gaikar

**College:** Sandip University

**Course:** B. Sc. Forensic Science (T. Y.)

**Objective:** Build a supervised machine-learning model that classifies network connections as normal traffic or one of four broad attack families: DoS, Probe, R2L and U2R.

This notebook uses the public NSL-KDD benchmark dataset.

## 1. Import Libraries\nThe project uses Pandas/NumPy for data processing, Scikit-learn for machine learning, and Matplotlib/Seaborn for visualization.

In [ ]:
import os\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import (accuracy_score, classification_report,\n                             confusion_matrix, precision_score, recall_score, f1_score)\n\nRANDOM_STATE = 42\nprint('Libraries imported successfully.')

## 2. Download the Dataset\nThe notebook downloads the NSL-KDD training and test files from a public GitHub mirror. The original dataset is documented by the Canadian Institute for Cybersecurity.

In [ ]:
DATA_DIR = Path('data')\nDATA_DIR.mkdir(exist_ok=True)\n\ntrain_url = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt'\ntest_url = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt'\n\ntrain_path = DATA_DIR / 'KDDTrain+.txt'\ntest_path = DATA_DIR / 'KDDTest+.txt'\n\nif not train_path.exists():\n    urllib.request.urlretrieve(train_url, train_path)\nif not test_path.exists():\n    urllib.request.urlretrieve(test_url, test_path)\n\nprint('Dataset files ready:', train_path, test_path)

In [ ]:
columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty']\n\ntrain_df = pd.read_csv(train_path, header=None, names=columns)\ntest_df = pd.read_csv(test_path, header=None, names=columns)\n\nprint('Training shape:', train_df.shape)\nprint('Testing shape:', test_df.shape)\ndisplay(train_df.head())

## 3. Label Engineering\nThe original dataset contains many individual attack names. For a practical academic NIDS, they are grouped into five classes: Normal, DoS, Probe, R2L and U2R.

In [ ]:
# Map the individual NSL-KDD attack names into five security classes.
attack_category = {}

dos_attacks = [
    "back", "land", "neptune", "pod", "smurf", "teardrop", "mailbomb",
    "processtable", "udpstorm", "apache2", "worm"
]
probe_attacks = [
    "ipsweep", "nmap", "portsweep", "satan", "mscan", "saint"
]
r2l_attacks = [
    "ftp_write", "guess_passwd", "imap", "multihop", "phf", "spy",
    "warezclient", "warezmaster", "named", "sendmail", "snmpgetattack",
    "snmpguess", "xlock", "xsnoop", "httptunnel"
]
u2r_attacks = [
    "buffer_overflow", "loadmodule", "perl", "rootkit", "ps", "sqlattack",
    "xterm"
]

for attack in dos_attacks:
    attack_category[attack] = "DoS"
for attack in probe_attacks:
    attack_category[attack] = "Probe"
for attack in r2l_attacks:
    attack_category[attack] = "R2L"
for attack in u2r_attacks:
    attack_category[attack] = "U2R"

def map_label(label):
    return "Normal" if label == "normal" else attack_category.get(label, "Other")

train_df["attack_category"] = train_df["label"].map(map_label)
test_df["attack_category"] = test_df["label"].map(map_label)

# Remove any unexpected labels rather than silently assigning them to a known attack class.
train_df = train_df[train_df["attack_category"] != "Other"].copy()
test_df = test_df[test_df["attack_category"] != "Other"].copy()

print("Training classes:")
print(train_df["attack_category"].value_counts())
print("\nTesting classes:")
print(test_df["attack_category"].value_counts())


## 4. Exploratory Data Analysis\nFirst we inspect the class balance. This is important because intrusion datasets can contain strongly imbalanced classes.

In [ ]:
plt.figure(figsize=(8, 5))\nsns.countplot(data=train_df, x='attack_category', order=train_df['attack_category'].value_counts().index)\nplt.title('Training Set: Network Traffic Classes')\nplt.xlabel('Class')\nplt.ylabel('Number of Records')\nplt.xticks(rotation=20)\nplt.tight_layout()\nplt.show()

## 5. Prepare Features and Target\nThe `label` and `difficulty` columns are not used as model inputs. The target is the grouped `attack_category`.

In [ ]:
feature_columns = [c for c in columns if c not in ['label', 'difficulty', 'attack_category']]\n\nX_train = train_df[feature_columns]\ny_train = train_df['attack_category']\nX_test = test_df[feature_columns]\ny_test = test_df['attack_category']\n\ncategorical_features = ['protocol_type', 'service', 'flag']\nnumeric_features = [c for c in feature_columns if c not in categorical_features]\n\npreprocessor = ColumnTransformer(\n    transformers=[\n        ('num', StandardScaler(), numeric_features),\n        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)\n    ]\n)\n\nprint('Numeric features:', len(numeric_features))\nprint('Categorical features:', len(categorical_features))

## 6. Train Model 1 — Logistic Regression\nLogistic Regression provides a simple baseline for multi-class classification.

In [ ]:
logistic_model = Pipeline(steps=[\n    ('preprocessor', preprocessor),\n    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=None))\n])\n\nlogistic_model.fit(X_train, y_train)\nlogistic_pred = logistic_model.predict(X_test)\n\nprint('Logistic Regression accuracy:', round(accuracy_score(y_test, logistic_pred), 4))\nprint(classification_report(y_test, logistic_pred, digits=4))

## 7. Train Model 2 — Random Forest\nRandom Forest is a strong tree-based baseline for tabular security data and also provides feature-importance estimates.

In [ ]:
rf_model = Pipeline(steps=[\n    ('preprocessor', preprocessor),\n    ('classifier', RandomForestClassifier(\n        n_estimators=200,\n        random_state=RANDOM_STATE,\n        n_jobs=-1,\n        class_weight='balanced_subsample'\n    ))\n])\n\nrf_model.fit(X_train, y_train)\nrf_pred = rf_model.predict(X_test)\n\nprint('Random Forest accuracy:', round(accuracy_score(y_test, rf_pred), 4))\nprint(classification_report(y_test, rf_pred, digits=4))

## 8. Compare the Models\nWe compare accuracy, weighted precision, weighted recall and weighted F1-score.

In [ ]:
results = pd.DataFrame({\n    'Model': ['Logistic Regression', 'Random Forest'],\n    'Accuracy': [\n        accuracy_score(y_test, logistic_pred),\n        accuracy_score(y_test, rf_pred)\n    ],\n    'Precision': [\n        precision_score(y_test, logistic_pred, average='weighted', zero_division=0),\n        precision_score(y_test, rf_pred, average='weighted', zero_division=0)\n    ],\n    'Recall': [\n        recall_score(y_test, logistic_pred, average='weighted', zero_division=0),\n        recall_score(y_test, rf_pred, average='weighted', zero_division=0)\n    ],\n    'F1': [\n        f1_score(y_test, logistic_pred, average='weighted', zero_division=0),\n        f1_score(y_test, rf_pred, average='weighted', zero_division=0)\n    ]\n})\n\ndisplay(results.round(4))

## 9. Confusion Matrix — Best Model\nThe model with the higher weighted F1-score is selected automatically for visualization.

In [ ]:
best_name = results.loc[results['F1'].idxmax(), 'Model']\nbest_pred = rf_pred if best_name == 'Random Forest' else logistic_pred\n\ncm = confusion_matrix(y_test, best_pred, labels=['Normal', 'DoS', 'Probe', 'R2L', 'U2R'])\n\nplt.figure(figsize=(8, 6))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n            xticklabels=['Normal', 'DoS', 'Probe', 'R2L', 'U2R'],\n            yticklabels=['Normal', 'DoS', 'Probe', 'R2L', 'U2R'])\nplt.title(f'Confusion Matrix — {best_name}')\nplt.xlabel('Predicted Class')\nplt.ylabel('True Class')\nplt.tight_layout()\nplt.show()

## 10. Random Forest Feature Importance\nIf Random Forest is selected, the notebook shows the most influential transformed features. This helps explain which network characteristics contribute most to the classifier.

In [ ]:
rf_preprocessor = rf_model.named_steps['preprocessor']\nrf_classifier = rf_model.named_steps['classifier']\nfeature_names = rf_preprocessor.get_feature_names_out()\n\nimportance_df = pd.DataFrame({\n    'Feature': feature_names,\n    'Importance': rf_classifier.feature_importances_\n}).sort_values('Importance', ascending=False).head(15)\n\nplt.figure(figsize=(10, 6))\nsns.barplot(data=importance_df, x='Importance', y='Feature')\nplt.title('Top 15 Random Forest Feature Importances')\nplt.tight_layout()\nplt.show()

## 11. Security Interpretation\nA high-performing NIDS classifier can support analysts by flagging suspicious traffic for further investigation. However, benchmark performance does not mean the model is ready for production. Real deployments require current traffic, drift monitoring, false-positive analysis, privacy controls, and testing against modern attack patterns.\n\n### Limitations\n- NSL-KDD is a benchmark dataset rather than a live modern network feed.\n- The dataset reflects older traffic and attack patterns.\n- Class imbalance and unseen attack behavior can affect generalization.\n- Accuracy alone is not enough for a production IDS; precision, recall and false-positive rates matter.

## 12. Conclusion\nThis project demonstrates a complete defensive cybersecurity ML workflow: dataset acquisition, preprocessing, label engineering, supervised model training, evaluation, visualization and security interpretation. The final model comparison is generated from the actual test-set results above.